# VietHandOCR Part 2: Baseline Evaluation

Welcome to **Part 2** of the VietHandOCR pipeline.
- **Previous Notebook**: [Part 1: Data Preparation & EDA](./01_Data_Preparation_and_EDA.ipynb)
- **Next Notebook**: [Part 3: Digital Image Processing (DIP) Pipeline](./03_Digital_Image_Processing.ipynb)

## Introduction
This notebook measures the Zero-shot performance of the pre-trained `vgg_transformer` model on the raw test set. This provides a Baseline to prove the effectiveness of the upcoming DIP steps and Fine-tuning.


In [ ]:
!pip install -q vietocr jiwer nltk


In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import jiwer
import nltk

nltk.download('punkt', quiet=True)

from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

def calculate_metrics(predictions, targets):
    '''Calculates CER, WER, Exact Match, and BLEU.'''
    cer = jiwer.cer(targets, predictions)
    wer = jiwer.wer(targets, predictions)
    exact_match = sum(1 for p, t in zip(predictions, targets) if p == t) / len(targets)
    
    bleu_scores = []
    for p, t in zip(predictions, targets):
        reference = [nltk.word_tokenize(t.lower())]
        candidate = nltk.word_tokenize(p.lower())
        if len(candidate) == 0:
            bleu_scores.append(0.0)
            continue
        try:
            bleu = nltk.translate.bleu_score.sentence_bleu(reference, candidate, weights=(0.5, 0.5))
            bleu_scores.append(bleu)
        except Exception:
            bleu_scores.append(0.0)
            
    avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0
    
    return {"CER": cer, "WER": wer, "Exact Match": exact_match, "BLEU": avg_bleu}


In [ ]:
def evaluate_baseline(test_txt_path, images_base_dir):
    config = Cfg.load_config_from_name('vgg_transformer')
    config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    predictor = Predictor(config)
    print("Loaded pre-trained vgg_transformer model successfully!")

    with open(test_txt_path, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')
        
    targets = []
    predictions = []
    results_detail = []
    
    print(f"Starting evaluation on {len(lines)} test images...")
    for line in tqdm(lines):
        if not line.strip(): continue
        parts = line.split('\t')
        if len(parts) != 2: continue
            
        rel_img_path, ground_truth = parts
        full_img_path = os.path.join(images_base_dir, rel_img_path)
        
        try:
            img = Image.open(full_img_path)
            pred = predictor.predict(img)
            
            targets.append(ground_truth)
            predictions.append(pred)
            
            results_detail.append({
                'image_path': rel_img_path,
                'ground_truth': ground_truth,
                'prediction': pred,
                'is_exact_match': ground_truth == pred
            })
        except Exception as e:
            print(f"Error processing {full_img_path}: {e}")
            
    metrics = calculate_metrics(predictions, targets)
    print("\n" + "="*40)
    print("🏆 BASELINE 1 RESULTS (Zero-shot)")
    print("="*40)
    for k, v in metrics.items():
        print(f"{k:<15}: {v:.4f}")
    print("="*40)
    
    df_results = pd.DataFrame(results_detail)
    df_results.to_csv('baseline1_error_analysis.csv', index=False, encoding='utf-8')
    print("Detailed results saved to 'baseline1_error_analysis.csv' for Error Analysis.")
    
    return metrics, df_results


In [ ]:
# Động bộ đường dẫn giống hệt file 01_Data_Preparation_and_EDA
base_dataset_path = next((os.path.join('/kaggle/input', d) for d in os.listdir('/kaggle/input') if os.path.isdir(os.path.join('/kaggle/input', d))), 'VietHandOCR_Datasets') if os.path.exists('/kaggle/input') else 'VietHandOCR_Datasets'

# File test.txt giả định nằm cùng thư mục hoặc bạn trỏ tay vào thư mục tương ứng trên Kaggle (ví dụ: '/kaggle/input/.../test.txt')
TEST_TXT_PATH = 'test.txt'

if not os.path.exists(TEST_TXT_PATH):
    print(f"Warning: {TEST_TXT_PATH} not found. Vui lòng chạy file 01 trước hoặc Add Data output của file 01 vào Kaggle.")
else:
    metrics, df_results = evaluate_baseline(TEST_TXT_PATH, base_dataset_path)
